<a href="https://colab.research.google.com/github/vuquocanh06/spacex-falcon9-capstone/blob/main/05_folium_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install folium

In [3]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(url)
print(spacex_df.columns.tolist())
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

['Flight Number', 'Date', 'Time (UTC)', 'Booster Version', 'Launch Site', 'Payload', 'Payload Mass (kg)', 'Orbit', 'Customer', 'Landing Outcome', 'class', 'Lat', 'Long']


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [4]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)

site_map

In [11]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5, tiles="CartoDB positron")

In [5]:
marker_color_map = {1: 'green', 0: 'red'}
spacex_df['marker_color'] = spacex_df['class'].map(marker_color_map)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [12]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5, tiles="CartoDB positron")

In [6]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map)

for _, row in spacex_df.iterrows():
    folium.Marker(
        [row['Lat'], row['Long']],
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=f"{row['Launch Site']} - {'Success' if row['class'] == 1 else 'Failed'}"
    ).add_to(marker_cluster)

site_map

In [7]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
MousePosition(position='topright', separator=' Long: ', empty_string='NaN',
              lng_first=False, num_digits=20, prefix='Lat:',
              lat_formatter=formatter, lng_formatter=formatter).add_to(site_map)
site_map

In [8]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [13]:
site_map = folium.Map(
    location=nasa_coordinate, zoom_start=5,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery"
)

In [14]:
launch_site_lat, launch_site_lon = launch_sites_df.iloc[0]['Lat'], launch_sites_df.iloc[0]['Long']

# Điền tọa độ thật của từng điểm gần bãi phóng đầu tiên
proximities = {
    "Coastline": (None, None),   # thay None bằng lat, lon
    "Railway":   (None, None),
    "Highway":   (None, None),
    "City":      (None, None),
}

for name, (lat, lon) in proximities.items():
    if lat is None:
        continue   # bỏ qua điểm chưa điền
    dist = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    folium.Marker(
        [lat, lon],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html=f'<div style="font-size: 12; color:#d35400;"><b>{name}: {dist:.2f} KM</b></div>')
    ).add_to(site_map)
    folium.PolyLine([[launch_site_lat, launch_site_lon], [lat, lon]], weight=1).add_to(site_map)
    print(name, round(dist, 2), "km")

site_map